# Chapter 15 — DPO Training

SFT teaches instruction-following, but not which responses humans prefer. RLHF aligns the model with human preferences — the secret sauce that made ChatGPT feel different from previous models.

In [ ]:
import torch
import torch.nn.functional as F

print("=== DPO (Direct Preference Optimization) ===\n")

torch.manual_seed(42)
batch_size = 4

pi_chosen = torch.randn(batch_size) * 0.5 - 2.0    # Log-probs from the policy model for preferred responses
pi_rejected = torch.randn(batch_size) * 0.5 - 3.0   # Log-probs from the policy model for rejected responses
ref_chosen = torch.randn(batch_size) * 0.5 - 2.5    # Log-probs from the frozen reference model for preferred responses (baseline)
ref_rejected = torch.randn(batch_size) * 0.5 - 2.8  # Log-probs from the frozen reference model for rejected responses (baseline)

log_ratio_chosen = pi_chosen - ref_chosen      # How much policy diverged from reference for chosen response
log_ratio_rejected = pi_rejected - ref_rejected  # How much policy diverged from reference for rejected response

print(f"{'Beta':>6s} | {'DPO Loss':>10s} | Interpretation")
print("-" * 48)
for beta in [0.05, 0.1, 0.2, 0.5, 1.0]:  # beta = KL penalty strength - higher = stay closer to reference model
    logits = beta * (log_ratio_chosen - log_ratio_rejected)  # DPO decision signal - positive means policy correctly prefers chosen
    loss = -F.logsigmoid(logits).mean()
    strength = "weak" if beta < 0.1 else "moderate" if beta < 0.3 else "strong"
    print(f"{beta:>6.2f} | {loss.item():>10.4f} | {strength} KL penalty")

beta = 0.1  # KL penalty strength - higher = stay closer to reference model
lr_c = log_ratio_chosen.clone().requires_grad_(True)
lr_r = log_ratio_rejected.clone().requires_grad_(True)
loss = -F.logsigmoid(beta * (lr_c - lr_r)).mean()
loss.backward()

print(f"\nGradient direction (beta={beta}):")
print(f"  d/d(chosen):   {lr_c.grad.mean():.4f} (negative -> increase chosen)")
print(f"  d/d(rejected): {lr_r.grad.mean():.4f} (positive -> decrease rejected)")

In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model=sft_model,
    ref_model=ref_model,
    train_dataset=preference_data,
    # {"prompt": "...", "chosen": "...", "rejected": "..."}
    args=DPOConfig(
        beta=0.1,
        learning_rate=5e-7,
        num_train_epochs=1,
    ),
)
trainer.train()

---

**Course:** Aprenda LLM | **Chapter 15**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.